# Practical 2: Data cleaning on customer_data.csv

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("customer_data.csv")
print("Original shape:", df.shape)

In [ ]:
# a) drop exact duplicate rows (e.g. C021 appears twice)
df = df.drop_duplicates()
print("After dropping duplicates:", df.shape)

In [ ]:
# b) "XXXX" placeholder -> NaN
df["Customer_Lifespan_Months"] = pd.to_numeric(
    df["Customer_Lifespan_Months"].replace("XXXX", np.nan), errors="coerce")

In [ ]:
# c) parse date column(s)
date_cols = [c for c in df.columns if "date" in c.lower()]
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors="coerce")

print("\nMissing values before filling:\n", df.isnull().sum())

In [ ]:
# d) fill missing values
for c in df.columns:
    if df[c].isnull().sum() == 0:
        continue
    if c in date_cols or pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].fillna(df[c].median())      # median (numbers / dates)
    else:
        df[c] = df[c].fillna(df[c].mode()[0])     # mode (text)
print("\nMissing values after filling:\n", df.isnull().sum())

In [ ]:
# e) label encoding (skip ID-like columns where every value is unique)
for c in df.select_dtypes(include="object").columns:
    if df[c].nunique() < len(df):
        df[c + "_enc"] = LabelEncoder().fit_transform(df[c])
print("\nAfter label encoding:\n", df.head())

In [ ]:
# f) outliers using box plot
num_cols = [c for c in df.select_dtypes(include=np.number).columns
            if not c.endswith("_enc")]
df[num_cols].plot(kind="box", subplots=True,
                  layout=(2, -(-len(num_cols) // 2)),
                  figsize=(12, 7), sharex=False, sharey=False)
plt.suptitle("Box plots - outlier detection")
plt.tight_layout()
plt.show()

In [ ]:
# count outliers with IQR rule
for c in num_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum()
    print(f"{c}: {n_out} outliers")